> **📦 Optional dependencies**
>
> This article uses features from the pdstools `adm` extra. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm]"`.


In [1]:
# --- IGNORE ---
import plotly.io as pio

pio.renderers.default = "notebook_connected"

# Example Prediction Studio Analysis

This is a small notebook to report and analyse Prediction Studio data on Predictions. The underlying data is from the Data-DM-Snapshot table that is used to populate the Prediction Studio screen with Prediction Performance, Lift, CTR etc.

Data can be exported from the **pyGetSnapshot** dataset in Pega Infinity from Dev Studio and Prediction Studio.

Datamart tables are described [here](https://docs.pega.com/bundle/platform/page/platform/decision-management/database-tables-monitoring-models.html).

## Raw data

First, we're going to load the raw data. The raw data is in a "long" format with e.g. test and control groups in separate rows.

In [2]:
from pathlib import Path
from pdstools import Prediction, demo_data

# path to dataset export here
# e.g. PR_DATA_DM_SNAPSHOTS.parquet
data_export = "<YOUR DATA HERE>"

if Path(data_export).exists():
    prediction = Prediction.from_ds_export(data_export)
else:
    # Notebook examples intentionally generate data; the Health Check app uses source="snapshot".
    prediction = demo_data(source="generated", days=60).prediction


## Prediction Data

The actual prediction data is in a "wide" format with separate fields for Test and Control groups. Also, it is only the "daily" snapshots and the numbers and date are formatted to be normal Polars types.

In [3]:
prediction.predictions.head().collect()

pyModelId,SnapshotTime,Positives,Negatives,ResponseCount,Performance,Positives_Test,Negatives_Test,ResponseCount_Test,Positives_Control,Negatives_Control,ResponseCount_Control,Positives_NBA,Negatives_NBA,ResponseCount_NBA,Class,ModelName,CTR,CTR_Test,CTR_Control,CTR_NBA,CTR_Lift,isValidPrediction
str,date,f64,f64,f64,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,bool
"""DATA-DECISION-REQUEST-CUSTOMER…",2026-09-17,999.0,39313.0,40312.0,0.640652,7.0,196.0,203.0,4.0,180.0,184.0,988.0,38937.0,39925.0,"""DATA-DECISION-REQUEST-CUSTOMER""","""PREDICTACTIONPROPENSITY""",0.024782,0.034483,0.021739,0.024746,0.586207,true
"""DATA-DECISION-REQUEST-CUSTOMER…",2026-09-18,959.0,38763.0,39722.0,0.638887,5.0,197.0,202.0,2.0,195.0,197.0,952.0,38371.0,39323.0,"""DATA-DECISION-REQUEST-CUSTOMER""","""PREDICTACTIONPROPENSITY""",0.024143,0.024752,0.010152,0.02421,1.438119,true
"""DATA-DECISION-REQUEST-CUSTOMER…",2026-09-19,973.0,37918.0,38891.0,0.683912,7.0,203.0,210.0,5.0,151.0,156.0,961.0,37564.0,38525.0,"""DATA-DECISION-REQUEST-CUSTOMER""","""PREDICTACTIONPROPENSITY""",0.025019,0.033333,0.032051,0.024945,0.04,true
"""DATA-DECISION-REQUEST-CUSTOMER…",2026-09-20,944.0,37586.0,38530.0,0.682767,5.0,176.0,181.0,4.0,206.0,210.0,935.0,37204.0,38139.0,"""DATA-DECISION-REQUEST-CUSTOMER""","""PREDICTACTIONPROPENSITY""",0.0245,0.027624,0.019048,0.024516,0.450276,true
"""DATA-DECISION-REQUEST-CUSTOMER…",2026-09-21,1026.0,40017.0,41043.0,0.630254,7.0,203.0,210.0,6.0,210.0,216.0,1013.0,39604.0,40617.0,"""DATA-DECISION-REQUEST-CUSTOMER""","""PREDICTACTIONPROPENSITY""",0.024998,0.033333,0.027778,0.02494,0.2,true


## Summary by Channel

Standard functionality exists to summarize the predictions per channel. Note that we do not have the prediction to channel mapping in the data (this is an outstanding product issue), so apply the implicit naming conventions of NBAD. For a specific customer, custom mappings can be passed into the summarization function.

In [4]:
prediction.summary_by_channel().collect()

Prediction,Channel,Direction,usesNBAD,isMultiChannel,DateRange Min,DateRange Max,Duration,Performance,Positives,Negatives,Responses,Positives_Test,Positives_Control,Positives_NBA,Negatives_Test,Negatives_Control,Negatives_NBA,usesImpactAnalyzer,ControlPercentage,TestPercentage,CTR,CTR_Test,CTR_Control,CTR_NBA,ChannelDirectionGroup,isValid,Lift
str,str,str,bool,bool,date,date,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,str,bool,f64
"""PREDICTACTIONPROPENSITY""","""Multi-channel""","""Multi-channel""",true,true,2026-09-17,2026-11-15,5097600,0.65899,58258.0,2.336535e6,2.394793e6,308.0,223.0,57727.0,11707.0,11772.0,2.313056e6,true,0.500878,0.501714,0.024327,0.025635,0.018591,0.024349,"""Other""",true,0.378867
"""PREDICTMOBILEPROPENSITY""","""Mobile""","""Inbound""",true,false,2026-09-17,2026-11-15,5097600,0.648722,63482.0,3.538076e6,3.601558e6,339.0,257.0,62886.0,17666.0,17421.0,3.502989e6,true,0.490843,0.499923,0.017626,0.018828,0.014538,0.017636,"""Mobile/Inbound""",true,0.29511
"""PREDICTOUTBOUNDCALLCENTERPROPE…","""Call Center""","""Outbound""",true,false,2026-09-17,2026-11-15,5097600,0.560419,26.0,214.0,240.0,1.0,0.0,0.0,0.0,3.0,0.0,false,75.0,25.0,0.108333,1.0,0.0,NaN,"""Call Center/Outbound""",false,inf
"""PREDICTOUTBOUNDEMAILPROPENSITY""","""E-mail""","""Outbound""",true,false,2026-09-17,2026-11-15,5097600,0.672,80148.0,5.958283e6,6.038431e6,413.0,315.0,79420.0,30424.0,29617.0,5.898242e6,true,0.495692,0.510679,0.013273,0.013393,0.010524,0.013286,"""E-mail/Outbound""",true,0.272633
"""PREDICTWEBPROPENSITY""","""Web""","""Inbound""",true,false,2026-09-17,2026-11-15,5097600,0.757273,254987.0,8.700291e6,8.955278e6,1345.0,850.0,252792.0,42995.0,43832.0,8.613464e6,true,0.498946,0.495127,0.028473,0.030334,0.019023,0.028512,"""Web/Inbound""",true,0.594558


## Prediction Trends

Summarization by default is over all time. You can pass in an argument to summarize by day, week or any other period as supported by the [Polars time offset string language](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.dt.offset_by.html).

This trend data can then easily be visualized.

In [5]:
fig = prediction.plot.performance_trend("1w")
fig.show()

In [6]:
fig = prediction.plot.lift_trend("1w")
fig.show()

In [7]:
fig = prediction.plot.ctr_trend("1w", facetting=False)
fig.show()

In [8]:
fig = prediction.plot.responsecount_trend("1w", facetting=False)
fig.show()